In [1]:
!pip install sentence-transformers numpy

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [3]:
class ResumeDB:
    def __init__(self):
        self.resumes = []
        self.vectors = []

    def add(self, resume, vector):
        self.resumes.append(resume)
        self.vectors.append(vector)

    def search(self, query_vector, top_k=3):
        scores = []

        for vec in self.vectors:
            sim = np.dot(vec, query_vector) / (
                np.linalg.norm(vec) * np.linalg.norm(query_vector)
            )
            scores.append(sim)

        top_idx = np.argsort(scores)[-top_k:][::-1]

        return [(self.resumes[i], scores[i]) for i in top_idx]

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')
db = ResumeDB()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
resumes = [
    "Python developer with experience in machine learning and data analysis",
    "Frontend developer skilled in HTML CSS JavaScript React",
    "Data scientist with expertise in deep learning and AI",
    "Backend developer with Java Spring Boot and SQL experience",
    "AI engineer with NLP and computer vision experience"
]

embeddings = model.encode(resumes)

for r, e in zip(resumes, embeddings):
    db.add(r, e)

print("Resumes stored successfully!")

Resumes stored successfully!


In [6]:
job_desc = "Looking for AI and machine learning engineer"

query_embedding = model.encode([job_desc])[0]

results = db.search(query_embedding, top_k=3)

print("Top Matching Resumes:\n")

for text, score in results:
    print(f"{text} (score: {score:.4f})")

Top Matching Resumes:

Data scientist with expertise in deep learning and AI (score: 0.5987)
AI engineer with NLP and computer vision experience (score: 0.5720)
Python developer with experience in machine learning and data analysis (score: 0.5128)


In [7]:
def explain_match(job_desc):
    query_embedding = model.encode([job_desc])[0]
    results = db.search(query_embedding, top_k=2)

    context = " ".join([res[0] for res in results])

    return f"""
Job Description: {job_desc}

Best Matching Candidates:
{context}

Reason:
These resumes match based on AI/ML skills and semantic similarity.
"""

In [8]:
print(explain_match("Need a machine learning engineer with Python"))


Job Description: Need a machine learning engineer with Python

Best Matching Candidates:
Python developer with experience in machine learning and data analysis Data scientist with expertise in deep learning and AI

Reason:
These resumes match based on AI/ML skills and semantic similarity.



In [10]:
while True:
    jd = input("\nEnter job description (type 'exit' to stop): ")

    if jd.lower() == "exit":
        break

    print(explain_match(jd))


Enter job description (type 'exit' to stop): exit
